In [1]:
from dotenv import load_dotenv
load_dotenv()

from neo4j_graphrag.retrievers import VectorRetriever
from neo4j_graphrag.indexes import create_vector_index

import random
from faker import Faker

import os
from datetime import datetime, timezone
from variables import COMPANIES, company_notes, hr_notes
from db_connection import db

fake = Faker('th_TH')



In [ ]:
from line_service import send_message, MessagePayload
send_message(MessagePayload(sender="Pachara", message="Hello"))

<coroutine object send_message at 0x0000019695E44E40>

## Add data

In [ ]:
def add_hr_contact(company_name: str, name: str, phone: str, status: str):
    created_at = datetime.now().strftime("%Y-%m-%d")
    
    cypher = """
    MERGE (c:Company {name: $company_name})
    
    MERGE (h:HR {name: $name, phone: $phone})
    ON CREATE SET 
        h.status = $status, 
        h.createdAt = $created_at
    ON MATCH SET 
        h.status = $status
        
    MERGE (c)-[r:HAS_HR]->(h)
    """
    
    with db.get_session() as session:
        session.run(
            cypher,
            company_name=company_name,
            name=name,
            phone=phone,
            status=status,
            created_at=created_at
        )
    
    print(f"✅ Added HR name: {name}, Company: {company_name}")

def generate_thai_phone_number(formatted: bool = True) -> str:
    prefixes = ['061', '062', '063', '064', '065', 
                '080', '081', '082', '083', '084', '085', '086', '087', '088', '089', 
                '090', '091', '092', '093', '094', '095', '096', '097', '098', '099']
    
    prefix = random.choice(prefixes)
    mid = f"{random.randint(0, 999):03d}"   
    last = f"{random.randint(0, 9999):04d}" 
    
    if formatted:
        return f"{prefix}-{mid}-{last}"      
    
    return f"{prefix}{mid}{last}" 


### Add company and hr

In [ ]:
for company_name in COMPANIES:
    hr_count = random.randint(1, 4)
    
    for _ in range(hr_count):
        phone = generate_thai_phone_number()
        status = random.choice(["active", "inactive"])
        
        add_hr_contact(company_name, fake.name(), phone, status)

### Add random note to company

In [ ]:
from ai import get_embedder

embedder = get_embedder()
with db.get_session() as session:
    created_at = datetime.now().strftime("%Y-%m-%d")
    notes_data = [
          {
            "text": text,
            "embedding": embedder.embed_query(text)
          }
          for text in company_notes
    ]

    inserted_notes = session.run("""
      MATCH (c:Company)
      WITH collect(c) AS all_companies, $notes_data AS notes
      WITH all_companies, notes, 
        CASE 
          WHEN size(all_companies) < size(notes) THEN size(all_companies)
          ELSE size(notes)
        END AS limit_size

      UNWIND range(0, limit_size - 1) AS idx
      
      WITH all_companies[idx] AS company, notes[idx] AS note_item
      
      CREATE (n:Note {
        message: note_item.text, 
        createdAt: $created_at, 
        embedding: note_item.embedding
      })
      CREATE (company)-[:HAS_NOTE]->(n)

      RETURN company.name AS company, note_item.text AS note_text
    """, notes_data=notes_data, created_at=created_at)
     
    for i in inserted_notes.data():
        print(f"{i['company']}:   {i['note_text']}")

In [ ]:
with db.get_session() as session:
    inserted_notes  = session.run("""MATCH (n:Note) Detach delete n""")

### Add random note to hr

In [ ]:
from datetime import datetime

with db.get_session() as session:
    created_at = datetime.now().strftime("%Y-%m-%d")
    
    formatted_notes = [
        {
            "text": text,
            "embedding": embedder.embed_query(text)
        }
        for text in hr_notes
    ]

    inserted_notes = session.run("""
      MATCH (h:HR)
      WITH collect(h) AS all_hr, $notes_data AS notes
      
      WITH all_hr, notes, 
           CASE 
             WHEN size(all_hr) < size(notes) THEN size(all_hr)
             ELSE size(notes)
           END AS limit_size

      UNWIND range(0, limit_size - 1) AS idx
      
      WITH all_hr[idx] AS hr, notes[idx] AS note_item
      
      CREATE (n:Note {
        message: note_item.text, 
        createdAt: $created_at, 
        embedding: note_item.embedding
      })
      CREATE (hr)-[:HAS_NOTE]->(n)
      
      RETURN hr.name AS hr, note_item.text AS note_text
    """, notes_data=formatted_notes, created_at=created_at)
  
    for i in inserted_notes.data():
        print(f"{i['hr']}:   {i['note_text']}")

In [ ]:
# def add_note(message: str, company_name=None, hr_name=None):
#     created_at = datetime.datetime.now().strftime("%Y-%m-%d")
#     if company_name:
#         cypher = """
#         MATCH (c:Company {name: $company_name})
#         CREATE (n:Note {message: $message, search_text: $search_text, created_at: $created_at, embedding: $embedding})
#         MERGE (c)-[:HAS_NOTE]->(n)
#         RETURN c.name as company_name
#         """
#         search_text = f"Company name: {company_name.lower()} Message: {message}"
#         embedding = embedder.embed_query(search_text)
    
#         with db.get_session() as session:
#          company = session.run(
#             cypher,
#             message=message,
#             search_text=search_text,
#             company_name=company_name,
#             created_at=created_at,
#             embedding=embedding
#          )
#          if len(company.data()) == 0: print(f"❌ Company name: {company_name} does not exist")
#          else: print(f"✅ Note added for Company name: {company_name}")
#         return
#     if hr_name:
#         cypher_hr = """
#         MATCH (h:HR {name: $hr_name})
#         CREATE (n:Note {message: $message, search_text: $search_text, created_at: $created_at, embedding: $embedding})
#         MERGE (h)-[:HAS_NOTE]->(n)
#         """
        
#         search_text = f"HR name: {hr_name.lower()} Message: {message}"
#         embedding = embedder.embed_query(search_text)
        
#         with db.get_session() as session:
#             hr = session.run(
#                 cypher_hr,
#                 message=message,
#                 search_text=search_text,
#                 hr_name=hr_name,
#                 created_at=created_at,
#                 embedding=embedding
#             )
#             if len(hr.data()) == 0: print(f"❌ HR name: {hr_name} does not exist")
#             else: print(f"✅ Note added for HR name: {hr_name}")
#         return
# add_note(message="มาไม่ตรงเวลา ถ้าขอมาครั้งหน้าอาจจะยัง", company_name="Siam Cement Group")

## Create Index

In [2]:
def show_index():
 with db.get_session() as session:
    a = session.run("show indexes")
    for i in a.data(): 
      print(f"type:{i['type']} name:{i['name']}")

show_index()

type:FULLTEXT name:fulltextIndex
type:LOOKUP name:index_1b9dcc97
type:LOOKUP name:index_460996c0
type:VECTOR name:vectorIndex


In [ ]:
with db.get_session() as session:
    session.run("""
CREATE FULLTEXT INDEX fulltextIndex IF NOT EXISTS
FOR (n:HR|Company|Note)
ON EACH [n.name, n.status, n.message]
""")

In [ ]:
cypher = """
DROP INDEX vectorIndex IF EXISTS
"""

with db.get_session() as session:
    session.run(cypher)
    print("✅ Dropped full-text index")

## Create Vector Index

In [ ]:
show_index()

In [ ]:
INDEX_NAME = "vectorIndex"

create_vector_index(
    db.driver,
    name=INDEX_NAME,
    label="Note",               
    embedding_property="embedding", 
    dimensions=1024,            
    similarity_fn="cosine",  
    neo4j_database=os.getenv("NEO4J_DATABASE")
)


In [ ]:
cypher = """
match (n:Note)
where n.embedding is not null
return n.embedding as embedding
"""

with db.get_session() as session:
    a = session.run(cypher)
    print(len(a.data()[0]['embedding']))


## Retrieve

In [ ]:
retriever = VectorRetriever(
    db.driver,
    neo4j_database=os.getenv("NEO4J_DATABASE"),
    index_name=INDEX_NAME,
    embedder=embedder,
    return_properties=['message'],
)

text =  "บริษัทไหนที่มีปัญหาเรื่องวิทยากรมาบรรยายสายบ้าง ขอชื่อ HR และเบอร์ติดต่อด้วย"
result = retriever.get_search_results(query_text=text, top_k=10)

for i in result.records:
    print(i.data())
    

## AI Assistant

In [ ]:
from ai import ai_assistant

question = "บริษัทไหนที่มีปัญหาเรื่องวิทยากรมาบรรยายสายบ้าง ขอชื่อ HR และเบอร์ติดต่อด้วย"

print("Answer:\n")
response = ai_assistant(question, top_k=10)
print(response.content[0].get('text', 'No response'))

print("\nToken:")
print(f"Input Tokens (Prompt): {response.usage_metadata['input_tokens']}")
print(f"Output Tokens (Completion): {response.usage_metadata['output_tokens']}")
print(f"Total Tokens: {response.usage_metadata['total_tokens']}")